In [1]:
import keras.backend as K
import os
import numpy as np
import pylab as plt
from keras.utils import to_categorical
from keras.models import Model
from keras.layers import Input
from keras.layers import LSTM
from keras.layers import Dense
from keras.layers.convolutional import Conv3D
from keras.layers.convolutional_recurrent import ConvLSTM2D
from keras.layers.normalization import BatchNormalization
from keras.models import load_model
from keras.models import load_model
from keras.callbacks import EarlyStopping
from keras.callbacks import ModelCheckpoint

import matplotlib.pyplot as plt
from keras.utils import to_categorical
from keras.regularizers import l2
from keras.backend import clip 
import math

## Reading Data 

In [2]:
data = np.load('data_with_frame_splits.npy')
data.shape

(657, 20, 128, 110, 1)

## MinMax Scaling 

In [3]:
from sklearn.preprocessing import minmax_scale

shape = data.shape
data = minmax_scale(data.ravel(), feature_range=(0,255)).reshape(shape)
data.shape

(657, 20, 128, 110, 1)

## Separating the Training and Testing data 

In [4]:
X1_precipitation = data[:600,:16,:,:,:]
X1_precipitation.shape

tem = np.zeros((600,1,128,110,1))
tem = tem.astype(int)

X2_precipitation = np.concatenate((tem, data[:600,16:19,:,:,:]), axis=1)
X2_precipitation = X2_precipitation.astype('uint8')
X2_precipitation.shape


y_precipitation = data[:600,16:20,:,:,:]
y_precipitation.shape
    
del tem

## 2 Layer Model Implmentation

In [5]:
"""
2-layer 
"""    
def define_models_2_precipitation(n_filter, filter_size):
    # define training encoder
    encoder_inputs = Input(shape=(None, 128, 110, 1))
    encoder_1 = ConvLSTM2D(filters = n_filter, kernel_size=filter_size, activation='relu', padding='same', return_sequences=True, return_state=True,
                           kernel_regularizer=l2(0.0005), recurrent_regularizer=l2(0.0005), bias_regularizer=l2(0.0005))
    encoder_2 = ConvLSTM2D(filters = n_filter, kernel_size=filter_size, activation='relu', padding='same', return_sequences=True, return_state=True,
                           kernel_regularizer=l2(0.0005), recurrent_regularizer=l2(0.0005), bias_regularizer=l2(0.0005))
    encoder_outputs_1, encoder_state_h_1, encoder_state_c_1 = encoder_1(encoder_inputs)
    encoder_outputs_2, encoder_state_h_2, encoder_state_c_2 = encoder_2(encoder_outputs_1)
    # define training decoder
    decoder_inputs = Input(shape=(None, 128, 110, 1))
    decoder_1 = ConvLSTM2D(filters=n_filter, kernel_size=filter_size, activation='relu', padding='same', return_sequences=True, return_state=True,
                           kernel_regularizer=l2(0.0005), recurrent_regularizer=l2(0.0005), bias_regularizer=l2(0.0005))
    decoder_2 = ConvLSTM2D(filters=n_filter, kernel_size=filter_size, activation='relu', padding='same', return_sequences=True, return_state=True,
                           kernel_regularizer=l2(0.0005), recurrent_regularizer=l2(0.0005), bias_regularizer=l2(0.0005))
    decoder_outputs_1, _, _ = decoder_1([decoder_inputs, encoder_state_h_1, encoder_state_c_1])
    decoder_outputs_2, _, _ = decoder_2([decoder_outputs_1, encoder_state_h_2, encoder_state_c_2])
    decoder_conv3d = Conv3D(filters=1, kernel_size=(1,1,64), activation='relu', padding='same', data_format='channels_last',
                            kernel_regularizer=l2(0.0005), bias_regularizer=l2(0.0005))
    decoder_outputs = decoder_conv3d(decoder_outputs_2)
    #clip(dec oder_outputs, 0, 255)
    
#    denselayer = Dense(1, activation='softmax')
#    decoder_outputs = denselayer(decoder_outputs)
    
    
    model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
    #print(model.summary(line_length=250))
    
    # define inference encoder
    encoder_model = Model(encoder_inputs, [encoder_state_h_1, encoder_state_c_1, encoder_state_h_2, encoder_state_c_2])
    
    # define inference decoder
    decoder_state_input_h_1 = Input(shape=(128,110,n_filter))
    decoder_state_input_c_1 = Input(shape=(128,110,n_filter))
    decoder_state_input_h_2 = Input(shape=(128,110,n_filter))
    decoder_state_input_c_2 = Input(shape=(128,110,n_filter))
    decoder_output_1, decoder_state_h_1_new, decoder_state_c_1_new = decoder_1([decoder_inputs, decoder_state_input_h_1, decoder_state_input_c_1])
    decoder_output_2, decoder_state_h_2_new, decoder_state_c_2_new = decoder_2([decoder_output_1, decoder_state_input_h_2, decoder_state_input_c_2])
    decoder_output = decoder_conv3d(decoder_output_2)
    #clip(decoder_output, 0, 255)
    
#    decoder_output = denselayer(decoder_output)
    
    decoder_model = Model([decoder_inputs , decoder_state_input_h_1 , decoder_state_input_c_1, decoder_state_input_h_2 , decoder_state_input_c_2],
                          [decoder_output, decoder_state_h_1_new, decoder_state_c_1_new, decoder_state_h_2_new, decoder_state_c_2_new])
    
    return model, encoder_model, decoder_model


## Model Parameters Implementation

In [6]:
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID" 
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [7]:
train_2_precipitation, infenc_2_precipitation, infdec_2_precipitation = define_models_2_precipitation(n_filter=64, filter_size=3)

train_2_precipitation.compile(loss='mse', optimizer='adam', metrics=['mse'])

#train_2_precipitation.fit([X1_precipitation,X2_precipitation], y_precipitation, batch_size=8, validation_split=0.25, epochs=1)
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=5)
#cp = ModelCheckpoint('model-mnist-2layer.h5', verbose=1, save_best_only=True)
filepath = "saved-modelTry-{epoch:02d}.h5"
cp = ModelCheckpoint(filepath, verbose=1, save_best_only=False,mode='max', period=5)
history_2_precipitation = train_2_precipitation.fit([X1_precipitation,X2_precipitation],
    y_precipitation, batch_size=6, validation_split=0.25, epochs=100, callbacks=[es,cp])



Epoch 1/100
75/75 [==============================] - 89s 1s/step - loss: 3932.8730 - mse: 3932.1125 - val_loss: 360.7039 - val_mse: 359.9371
Epoch 2/100
75/75 [==============================] - 88s 1s/step - loss: 172.8904 - mse: 172.1226 - val_loss: 141.2276 - val_mse: 140.4592
Epoch 3/100
75/75 [==============================] - 87s 1s/step - loss: 123.0575 - mse: 122.2885 - val_loss: 45.4817 - val_mse: 44.7122
Epoch 4/100
75/75 [==============================] - 88s 1s/step - loss: 45.2180 - mse: 44.4482 - val_loss: 38.5443 - val_mse: 37.7743
Epoch 5/100
75/75 [==============================] - ETA: 0s - loss: 28.3031 - mse: 27.5329
Epoch 00005: saving model to saved-modelTry-05.h5
75/75 [==============================] - 88s 1s/step - loss: 28.3031 - mse: 27.5329 - val_loss: 18.8523 - val_mse: 18.0819
Epoch 6/100
75/75 [==============================] - 87s 1s/step - loss: 30.3072 - mse: 29.5368 - val_loss: 28.4215 - val_mse: 27.6509
Epoch 7/100
75/75 [=============================

In [8]:
train_2_precipitation.save('dump.h5')